# 01: Baseline: Text Clustering as Classification with a Local LLM

**Goal of this notebook:** reproduce the original method of Huang & He, *Text Clustering as Classification with LLMs*, with one change: the paid GPT-3.5 API is replaced with **Qwen 2.5 (7B)** running locally through Ollama on a consumer GPU (RTX 4060, 8 GB).

**Why start with a baseline?** Every improvement we make later must be measured against something. This notebook gives that reference score, using the original pipeline with no changes to the method itself.

**The original method in two stages:**
1. **Label generation:** the LLM reads the texts in batches of 15 and proposes category names, then merges similar names into one final list.
2. **Classification:** the LLM reads each text and picks one label from that list. Texts with the same label form a cluster.

**Steps:** 0 Setup → 1 Example labels → 2 Generate labels → 2b/2c Check the merge → 3 Classify → 4 Evaluate → 5 Save → Findings

**Safe to Run All:** each slow step is skipped if its output file already exists (`REUSE = True`). Set `REUSE = False` to run everything again from scratch.

## 0. Setup
The notebook lives in `/notebooks`, but the scripts and data are in the main project folder, so we move there first.

In [ ]:
import os, sys
# Run from the main project folder even though the notebook is in /notebooks
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json, time, math, urllib.request
import pandas as pd

DATA = "massive_scenario"      # dataset folder inside ./dataset
MODEL = "qwen2.5:7b"           # local LLM in Ollama
SIZE = "small"                 # the paper uses the small split
REUSE = True                   # True: skip slow steps whose output already exists
RERUN_MERGE = False            # True: redo the merge check in 2b and overwrite the label list
OUT = "./generated_labels"
os.makedirs(OUT, exist_ok=True)
os.makedirs("./results", exist_ok=True)

def out_file(name):
    return f"{OUT}/{DATA}_{SIZE}_{name}.json"

ran = {"labelgen": False, "classify": False}   # tracks what was actually run in this session
print("Working folder:", os.getcwd())

### Check the data and the LLM
Before spending time on the LLM, we confirm the dataset file exists and Ollama is running with the model.

In [ ]:
data_file = f"./dataset/{DATA}/{SIZE}.jsonl"
assert os.path.exists(data_file), f"Dataset not found: {data_file}"
with open(data_file, encoding="utf-8") as f:
    n_texts = sum(1 for _ in f)
print(f"{DATA}: {n_texts} texts")

try:
    tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=5))
    models = [m["name"] for m in tags["models"]]
    print("Ollama is running. Models:", models)
    assert any(m.startswith(MODEL) for m in models), f"Run: ollama pull {MODEL}"
except Exception as e:
    print("Ollama problem:", e)

## 1. Select example labels (20% of the true labels)
**Why:** the original method is not fully unsupervised. It shows the LLM 20% of the true category names as examples, so the generated labels follow the same style. We keep this exactly as in the paper.

**One fix we made:** the original code used `random.choices`, which can pick the same label twice, and set no random seed. We use `random.sample` with a fixed seed, so every run picks the same distinct labels and results are reproducible.

In [ ]:
!python select_part_labels.py
with open(f"{OUT}/chosen_labels.json", encoding="utf-8") as f:
    chosen = json.load(f)
print("Example labels given to the LLM:", chosen[DATA])

## 2. Generate labels with the LLM
**What happens:** the texts are shuffled and sent in batches of 15, about `texts / 15` LLM calls (≈199 for massive_scenario). For each batch, the LLM either reuses an existing label or proposes a new one. At the end, **one** extra call asks the LLM to merge similar labels.

Skipped if the output already exists and `REUSE = True`.

In [ ]:
if REUSE and os.path.exists(out_file("llm_generated_labels_after_merge")):
    print("Labels already generated: reusing them (set REUSE = False to regenerate).")
    time_labelgen = 0
else:
    t0 = time.time()
    !python label_generation.py --data {DATA} --model {MODEL}
    time_labelgen = time.time() - t0
    ran["labelgen"] = True
    print(f"Label generation time: {time_labelgen/60:.1f} min")

### Inspect the generated labels
**What we check:** how many labels the LLM produced compared with the 18 true clusters. If the number is much larger, the clusters will be split into near-duplicates and the scores will drop.

In [ ]:
def load(name):
    with open(out_file(name), encoding="utf-8") as f:
        return json.load(f)
true_labels = load("true_labels")
before = load("llm_generated_labels_before_merge")
after = load("llm_generated_labels_after_merge")
print(f"True clusters: {len(true_labels)} | LLM labels before merge: {len(before)} | after merge: {len(after)}")
print("\nLLM labels after merge:\n", sorted(after))

### What we found in our run
- True clusters: **18**. LLM labels before merge: **76**. After merge: **76**.
- The merge changed **nothing**, and obvious duplicates remained: `alarm` / `alarms` / `alarms_reminders`, `event` / `events` / `upcoming_events`, `query` / `query_info` / `query_information`.

**Decision:** before spending ~30 minutes on classification, find out *why* the merge failed. It was either a parsing error (the answer was not valid JSON, so the code silently kept the original list) or a weak merge by the model.

## 2b. Debug the merge step
This cell reruns **only** the merge call and prints Qwen's raw answer, so we can see which of the two problems it is.

It only runs when `RERUN_MERGE = True`, because the answer can vary slightly between runs and we don't want Run All to overwrite the saved label list.

In [ ]:
if RERUN_MERGE:
    import importlib, label_generation as lg
    from types import SimpleNamespace
    importlib.reload(lg)
    lg.MODEL = MODEL
    client = lg.ini_client(SimpleNamespace(base_url="http://localhost:11434/v1", api_key="ollama"))
    raw = lg.chat(lg.prompt_construct_merge_label(before), client)
    print("RAW ANSWER:\n", raw)
    parsed = lg.parse_json(raw)
    merged = [l for v in parsed.values() for l in (v if isinstance(v, list) else [v])]
    print(f"\nParsed OK: {len(merged)} labels")
    print(sorted(merged))
else:
    merged = after
    print(f"Skipped (RERUN_MERGE = False). Using the saved list: {len(merged)} labels.")

### What we found
- Qwen's answer **was valid JSON**, so parsing was not the problem.
- But the model **barely merged anything: 76 → 71 labels.** The duplicates listed above were still separate.

**Conclusion:** the original pipeline relies on a strong model (GPT-3.5) to merge 76 labels in a single call. A small 7B local model can't do this reliably. This is a real weakness of the method in low-resource settings, and it becomes the first target of our improvement in notebook 02.

**Decision for the baseline:** keep Qwen's own merge (71 labels) as the honest baseline. We do not fix it here, because the baseline must reflect the original method.

## 2c. Save the merged labels
The 71 labels from Qwen's merge are saved as the label list that Step 3 classifies into.

In [ ]:
if RERUN_MERGE:
    with open(out_file("llm_generated_labels_after_merge"), "w", encoding="utf-8") as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(merged)} labels.")
after = merged
print(f"Labels used for classification: {len(after)}")

## 3. Classify every text into one label
**What happens:** for each of the 2,974 texts, the LLM receives the full label list and the text, and returns one label. That's **one LLM call per text**, which makes this the slowest and most expensive step of the method. In our run it took about **26 minutes** on the RTX 4060 (~2 texts per second).

If the LLM's answer does not exactly match a label in the list, the text goes to an **"Unsuccessful"** group.

Skipped if the output already exists and `REUSE = True`.

In [ ]:
if REUSE and os.path.exists(out_file("find_labels")):
    print("Classification already done: reusing it (set REUSE = False to rerun).")
    time_classify = 0
else:
    t0 = time.time()
    !python given_label_classification.py --data {DATA} --model {MODEL}
    time_classify = time.time() - t0
    ran["classify"] = True
    print(f"Classification time: {time_classify/60:.1f} min")

## 4. Evaluate
We compare the predicted clusters with the true categories using the three standard clustering metrics from the paper:
- **ACC:** accuracy after matching each predicted cluster to its best true cluster (Hungarian algorithm).
- **NMI (Normalized Mutual Information):** how much information the clustering shares with the true labels (0 to 1).
- **ARI (Adjusted Rand Index):** agreement on which pairs of texts are grouped together, corrected for chance.

As in the original `evaluate.py`, the "Unsuccessful" texts count as one extra cluster.

In [ ]:
import importlib, evaluate as ev
importlib.reload(ev)
gold = ev.load_data("./dataset/", DATA, SIZE == "large")
with open(out_file("find_labels"), encoding="utf-8") as f:
    pred_dict = json.load(f)
y_true_raw = ev.get_labels(gold)
y_pred_raw = ev.get_predict_labels(gold, pred_dict)
y_true_raw = y_true_raw[:len(y_pred_raw)]
y_true, k_true = ev.convert_label_to_ids(y_true_raw)
y_pred, k_pred = ev.convert_label_to_ids(y_pred_raw)
scores = {k: round(float(v), 4) for k, v in ev.clustering_score(y_true, y_pred).items()}
unsuccessful = len(pred_dict.get("Unsuccessful", []))
print(scores)
print(f"Texts the LLM could not classify: {unsuccessful}")

## 5. Save the results
Each run is stored as one row in `results/results.csv`, the shared table that all notebooks add to. That way the final comparison builds itself.

If nothing was rerun in this session and a baseline row already exists, we keep the existing row, so the original timings are not overwritten with zeros.

In [ ]:
path = "./results/results.csv"
df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
exists = (not df.empty) and ((df["method"] == "baseline") & (df["dataset"] == DATA) & (df["model"] == MODEL)).any()

if exists and not (ran["labelgen"] or ran["classify"]):
    print("Baseline row already saved and nothing was rerun: keeping the existing row.")
else:
    row = {
        "method": "baseline", "dataset": DATA, "model": MODEL, "n_texts": n_texts,
        "true_clusters": k_true, "predicted_clusters": k_pred, **scores,
        "unsuccessful": unsuccessful,
        "llm_calls": math.ceil(n_texts / 15) + 1 + n_texts,   # label generation + merge + classification
        "time_min": round((time_labelgen + time_classify) / 60, 1),
        "date": time.strftime("%Y-%m-%d %H:%M"),
    }
    if exists:
        df = df[~((df["method"] == "baseline") & (df["dataset"] == DATA) & (df["model"] == MODEL))]
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(path, index=False)
    print("Saved.")
df

### Predicted cluster sizes
This chart shows how the texts were spread across the LLM's labels. It's the clearest way to see *why* the scores are what they are.

In [ ]:
import matplotlib.pyplot as plt
sizes = {k: len(v) for k, v in pred_dict.items() if v}
pd.Series(sizes).sort_values().plot.barh(figsize=(7, max(3, len(sizes) * 0.22)))
plt.title(f"Predicted cluster sizes: {DATA}")
plt.xlabel("Number of texts"); plt.tight_layout(); plt.show()

## Findings from the baseline (our run, 7 Nov 2026)

| Metric | Score |
|---|---|
| ACC | 0.399 |
| NMI | 0.558 |
| ARI | 0.320 |
| Predicted clusters | 71 (true: 18) |
| Unsuccessful texts | 175 (5.9%) |
| LLM calls | 3,174 |
| Time | ~26 min (classification) |

**Three causes of the low scores, visible in the chart:**
1. **Too many overlapping labels.** The weak merge left near-duplicates (`alarm`, `alarms_reminders`, `alarms_reminders_time`, `reminder`; `music`, `play_music`, `personal_music`, `audio_play`). One true topic gets split across several labels, and ACC can only credit one of them.
2. **Catch-all labels** (`miscellaneous`, `other`, `request`, `information_request`) absorb texts that belong to real topics.
3. **Failed answers:** 175 texts were lost because Qwen answered with a label spelled slightly differently from the list.

**Next step: notebook 02** fixes each cause with one targeted change, reuses the *same* generated labels so the comparison is fair, and measures the effect of each fix.